# 03 — Preprocesamiento y Limpieza
**Fase 2 · CRISP-DM** | Proyecto: Predicción de Stroke

UNAM · Facultad de Ingeniería · Minería de Datos · Semestre 2027-1

---

Operaciones aplicadas:
1. Eliminación de `id` (sin valor predictivo)
2. Eliminación del único registro con `gender='Other'`
3. Imputación de `bmi` con la **mediana**
4. `smoking_status='Unknown'` se conserva como categoría propia
5. **One-hot encoding** de variables categóricas
6. **StandardScaler** sobre variables numéricas
7. **SMOTE** para balancear clases en el conjunto de entrenamiento

Salida:
- `data/processed_train.csv`: entrenamiento balanceado con SMOTE (para el ajuste final)
- `data/processed_train_unbalanced.csv`: entrenamiento **sin** SMOTE (para la validación cruzada, donde SMOTE se aplica dentro de cada fold)
- `data/processed_test.csv`: prueba con la distribución real de clases

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

DATA_DIR     = os.path.join(os.getcwd(), "data")
CSV_PATH     = os.path.join(DATA_DIR, "healthcare-dataset-stroke-data.csv")
RANDOM_STATE = 42
TEST_SIZE    = 0.20

df = pd.read_csv(CSV_PATH)
print(f"Dataset original: {df.shape}")

Dataset original: (5110, 12)


## 3.1 Limpieza básica

In [2]:
# Eliminar columna id
df = df.drop(columns=["id"])

# Eliminar registro con gender='Other' (n=1)
df = df[df["gender"] != "Other"].reset_index(drop=True)

# Imputar bmi con la mediana
bmi_median = df["bmi"].median()
df["bmi"] = df["bmi"].fillna(bmi_median)
print(f"bmi imputado con mediana = {bmi_median:.2f}")
print(f"Dataset tras limpieza: {df.shape}")

# Verificar que no quedan nulos
assert df.isnull().sum().sum() == 0, "Aún hay valores faltantes"
print("Sin valores faltantes ✓")

bmi imputado con mediana = 28.10
Dataset tras limpieza: (5109, 11)
Sin valores faltantes ✓


## 3.2 One-hot encoding

In [3]:
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(f"Columnas tras encoding: {df_enc.shape[1]}")
df_enc.head(2)

Columnas tras encoding: 21


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,ever_married_No,ever_married_Yes,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,False,True,False,True,...,False,True,False,False,False,True,False,True,False,False
1,61.0,0,0,202.21,28.1,1,True,False,False,True,...,False,False,True,False,True,False,False,False,True,False


## 3.3 División train / test (80 / 20 estratificado)

In [4]:
X = df_enc.drop(columns=["stroke"])
y = df_enc["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Distribución y_train: {y_train.value_counts().to_dict()}")

Train: 4087 muestras | Test: 1022 muestras
Distribución y_train: {0: 3888, 1: 199}


## 3.4 Estandarización (StandardScaler)

In [5]:
num_cols = ["age", "avg_glucose_level", "bmi"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])
print("Estandarización aplicada ✓")

Estandarización aplicada ✓


## 3.5 SMOTE — balanceo de clases en entrenamiento

In [6]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Distribución y_train ANTES de SMOTE: {y_train.value_counts().to_dict()}")
print(f"Distribución y_train DESPUÉS de SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")

Distribución y_train ANTES de SMOTE: {0: 3888, 1: 199}
Distribución y_train DESPUÉS de SMOTE: {0: 3888, 1: 3888}


## 3.6 Guardar conjuntos procesados

In [7]:
train_df = X_train_res.copy()
train_df["stroke"] = y_train_res
test_df  = X_test.copy()
test_df["stroke"]  = y_test.values

train_path = os.path.join(DATA_DIR, "processed_train.csv")
test_path  = os.path.join(DATA_DIR, "processed_test.csv")

# Entrenamiento sin balancear: la validación cruzada de 04_modeling aplica SMOTE
# dentro de cada fold para que los folds de validación no contengan muestras sintéticas.
train_unbal_df = X_train.copy()
train_unbal_df["stroke"] = y_train.values
train_unbal_path = os.path.join(DATA_DIR, "processed_train_unbalanced.csv")
train_unbal_df.to_csv(train_unbal_path, index=False)
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path,   index=False)
print(f"Guardado: {train_path}")
print(f"Guardado: {test_path}")
print(f"Guardado: {train_unbal_path}")

Guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\processed_train.csv
Guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\processed_test.csv
Guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\processed_train_unbalanced.csv


---
✅ **Fase de preprocesamiento completada.**  
Continúa en `04_modeling.ipynb`.